# LLMs and contexts

Oh boy. This notebook probably won't be properly annotated any time soon. It took way too long to run.

In [ ]:
import requests
import pandas as pd
import glob
import time
import re
from pathlib import Path

In [ ]:
kwic_folder = "/Users/inqfinity/mip/newdecades_kwic_outputs"

files = glob.glob(f"{kwic_folder}/democratic_*_kwic_top50_bigrams.csv")

dfs = []

for file in files:
    temp = pd.read_csv(file)
    temp["source_file"] = Path(file).name
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print(df.shape)
print(df.columns.tolist())

(33275, 13)
['target_word', 'period', 'year', 'bigram', 'party', 'speaker', 'major_heading', 'speech_id', 'left_context', 'match', 'right_context', 'category_code', 'source_file']


In [4]:
def build_context(row):

    left = str(row["left_context"]) if pd.notna(row["left_context"]) else ""
    match = str(row["match"]) if pd.notna(row["match"]) else ""
    right = str(row["right_context"]) if pd.notna(row["right_context"]) else ""

    return f"{left} [{match}] {right}"

df["context_for_llm"] = df.apply(build_context, axis=1)
df[["bigram", "context_for_llm"]].head()


,bigram,context_for_llm
0,democratic right,par rather one boys one mr speaker rest new me...
1,democratic society,stansted roskill commission also rule stansted...
2,democratic right,edens ideal propertyowning democracy repeat to...
3,democratic party,paul johnson lead socialist journalist former ...
4,democratic socialist,many case fear totally disappeared way country...


In [5]:
prompt = """
You are presented with bigrams related to "democratic" that are derived from a corpus of lemmatized UK parliamentary speeches, as well as words surrounding where they occured. Each row is one bigram occurence.

Tag bigrams based on these definitions and assign one category number only. Use both the bigram and the context obtained by KWIC to assing one category.

1. Party affiliation: when bigrams referred to UK parliamentary party names, for parties between 1980-2019 (e.g. democratic unionist)
2. International divisions: where "democratic" is used as a reference to countries (e.g. [German] Democratic Republic)
3. Constitutional issues: referring to state institutions and political questions (e.g. democratic structures)
4. Democratic principles: democratic principles for broad ideas and categories (e.g. democratic legitimacy)
5. Procedures and practices: procedures and practices when linked to specific processes (e.g. democratic vote)
6. Societal issues: societal issues when discussing broad societal topics (e.g. democratic community)

Note that bigrams and their surrounding words have already been lemmatized and isolated out of the full context of their original speeches. For your reasoning, pick a maximum of 3 words from the context or surrounding words that support your reasoning for labelling the bigram with a certain category.

Reply in this exact format and nothing else:
Category: [single digit 1-6]
Reason: [3 words max]
"""

In [ ]:
def parse_response(raw):

    raw = raw.strip()

    code_match = re.search(r"Category:\s*([1-6])", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)
    
    code = code_match.group(1) if code_match else "NA"
    reason = reason_match.group(1).strip() if reason_match else "NA"

    if reason != "NA":
        reason = " ".join(reason.split()[:5])

    return code, reason

In [7]:
def tag_with_model(model_name, bigram, context):
    full_prompt = f"""
{prompt}

Bigram:
{bigram}

KWIC Context:
{context}
"""
    
    response = requests.post("http://localhost:11434/api/generate", json={"model": model_name, "prompt": full_prompt, "stream": False, "options": { "temperature": 0}})
    raw = response.json()["response"]
    return parse_response(raw)

In [ ]:
#testing for 5 entires first!!
for i in range(5):

    row = df.iloc[i]

    print("\nBigram:")
    print(row["bigram"])

    print("Context:")
    print(row["context_for_llm"])

    result1 = tag_with_model("llama3", row["bigram"], row["context_for_llm"])
    print("Result Llama:")
    print(result1)

    result2 = tag_with_model("mistral", row["bigram"], row["context_for_llm"])
    print("Result Mistral:")
    print(result2)


Bigram:
democratic right
Context:
par rather one boys one mr speaker rest new member old part goodly company heirs great tradition exercise behalf represent [democratic right] freedom take perhaps grant long enjoyment country envy many people many land like air breathe little noticed presence value beyond
Result:
('5', 'democratic right freedom')

Bigram:
democratic society
Context:
stansted roskill commission also rule stansted yet people find stansted still one lead contender third london airport find mysterious meant [democratic society] hope government approach matter open mind base judgment cost factor grateful call early debate welcome general tone content gracious speech
Result:
('6', 'democratic, society, people')

Bigram:
democratic right
Context:
edens ideal propertyowning democracy repeat today floor devil mean mean u property right participate democracy somehow less human few citizen [democratic right] property kind think underlies strategy municipal asset strip involve wh

In [9]:
mistral_codes = []
mistral_reasons = []

llama_codes = []
llama_reasons = []

for i, row in df.iterrows():

    bigram = row["bigram"]
    context = row["context_for_llm"]

    m_code, m_reason = tag_with_model("mistral", bigram, context)

    l_code, l_reason = tag_with_model("llama3", bigram, context)

    mistral_codes.append(m_code)
    mistral_reasons.append(m_reason)

    llama_codes.append(l_code)
    llama_reasons.append(l_reason)

    print(f"[{i+1}/{len(df)}] " f"{bigram} | " f"Mistral={m_code} | " f"Llama={l_code}")

    time.sleep(0.3)

[1/33275] democratic right | Mistral=5 | Llama=5
[2/33275] democratic society | Mistral=6 | Llama=6
[3/33275] democratic right | Mistral=5 | Llama=5
[4/33275] democratic party | Mistral=1 | Llama=1
[5/33275] democratic socialist | Mistral=4 | Llama=1
[6/33275] proper democratic | Mistral=4 | Llama=3
[7/33275] democratic socialism | Mistral=4 | Llama=4
[8/33275] democratic state | Mistral=3 | Llama=3
[9/33275] democratic system | Mistral=3 | Llama=3
[10/33275] democratic process | Mistral=5 | Llama=5
[11/33275] democratic election | Mistral=5 | Llama=5
[12/33275] democratic right | Mistral=5 | Llama=6


KeyboardInterrupt: 

In [ ]:
df["category_code_mistral_kwic"] = mistral_codes
df["reason_mistral_kwic"] = mistral_reasons

df["category_code_llama_kwic"] = llama_codes
df["reason_llama_kwic"] = llama_reasons

In [ ]:
output_path = "/Users/inqfinity/mip/democratic_kwic_occurrences_tagged.csv"

df.to_csv(output_path, index=False)

print("Saved:")
print(output_path)

In [ ]:
summary = (df.groupby(["period", "category_code_llama_kwic"]).size().reset_index(name="count"))

summary